[Food concept phrase]  
    ├── qualifiers（lean / extra lean / omega-3）
    ├── source（beef / chicken / turkey）
    ├── preparation（canned / cooked / breaded）
    └── context（as a main dish / incl yolk / with skin）

build_ffq_item_rds()
│
├─ read_ffq_item_map_txt()     ← ① 读 FFQ item 原始 txt（对齐你 R 代码）
│
├─ mine_phrases()              ← ② 从整列 item 自动学习 food phrases
│
├─ for each item:
│    ├─ merge_phrases()        ← ③ 把 hot dog → hot_dog（防止被拆）
│    ├─ spaCy parse            ← ④ 依存句法解析
│    ├─ extract_entity_and_modifiers()
│    └─ combine_entity_modifiers()
│
└─ write_rds()                 ← ⑤ 输出 RDS，R 里直接用

In [15]:
from __future__ import annotations
import re
import math
from collections import Counter
from typing import List, Dict, Tuple, Set
import pandas as pd
import numpy as np
import pyreadr
import spacy
from rapidfuzz import process, fuzz
from scipy.optimize import linear_sum_assignment

In [16]:
from pathlib import Path
import os
import yaml

# project root = first parent folder containing `.here` (works whatever the cwd is)
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".here").exists())

def resolve(p: str) -> Path:
    p = Path(os.path.expanduser(p))
    return p if p.is_absolute() else (PROJECT_ROOT / p).resolve()

paths = {k: resolve(v) for k, v in yaml.safe_load(open(PROJECT_ROOT / "config" / "paths.yml")).items()}

# FFQ data
ffq_path = paths["ffq_raw_dir"] / "perls9.scn1.csv"
# FFQ item dictionary
ffq_item_path = paths["ffq_raw_dir"] / "perls9.scn1.csv.label.doc"
# exact food item and portion asked in the FFQ
portion_path = PROJECT_ROOT / "data_raw" / "master_food_table.csv"
output_rds_path = str(PROJECT_ROOT / "cache" / "ffq_item_map_parsed.rds")

In [17]:
# =========================================================
# 1) IO: read var=item txt (same as your R function)
# =========================================================
def read_ffq_item_map_txt(path: str, encoding: str = "utf-8") -> pd.DataFrame:
    with open(path, "r", encoding=encoding, errors="replace") as f:
        lines = f.read().splitlines()

    lines = [ln.strip() for ln in lines]                      # trimws
    lines = [ln for ln in lines if ln]                        # remove empty
    lines = [ln for ln in lines if not ln.startswith("#")]    # remove comment rows
    lines = [ln for ln in lines if "=" in ln]                 # keep only '=' rows

    vars_, items_ = [], []
    for ln in lines:
        var, item = ln.split("=", 1)                          # split once
        vars_.append(var.strip())
        items_.append(item.strip())

    return pd.DataFrame({"var": vars_, "item": items_})


In [18]:
ffq_item = read_ffq_item_map_txt(ffq_item_path)
ffq_item

,var,item
0,id,ID Number
1,sug,Sugar in Teaspoons/day
2,blank2,filler
3,multvit,Currently take multi-vitamin?
4,mulfrq,How many multi-vit per week?
...,...,...
249,as.sp,"Type of art sweetener, Splenda"
250,as.eq,"Type of art sweetener, Equal"
251,as.ns,"Type of art sweetener, NutraSweet"
252,as.sl,"Type of art sweetener, Sweet N Low"


In [19]:
STOPWORDS = {"and","or","with","without","incl","include","excluding","exclude","fresh","frozen","type"}

def normalize_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.lower().strip()
    s = s.replace("&", " and ")
    s = re.sub(r"[/\-]", " ", s)
    s = re.sub(r"[()]", " ", s)                 # keep words but remove brackets
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    toks = [t for t in s.split() if t and t not in STOPWORDS]
    return " ".join(toks)

def one_to_one_fuzzy_merge(
    ffq_df: pd.DataFrame,
    map_df: pd.DataFrame,
    ffq_item_col: str = "item",
    map_item_col: str = "food_item",
    keep_cols: list[str] = ["food_group", "portion"],
    scorer = fuzz.token_set_ratio,
    min_score: int = 85,
    top_k: int = 30,
) -> pd.DataFrame:
    """
    1-1 global matching using Hungarian algorithm (max total similarity).
    - min_score: pairs below this are treated as unavailable
    - top_k: for speed, only consider top_k candidates per ffq item
    """
    ffq = ffq_df.copy()
    mapping = map_df.copy()

    ffq["_key"] = ffq[ffq_item_col].apply(normalize_text)
    mapping["_key"] = mapping[map_item_col].apply(normalize_text)

    a = ffq["_key"].tolist()
    b = mapping["_key"].tolist()
    n, m = len(a), len(b)
    N = max(n, m)

    BIG = 10_000
    cost = np.full((N, N), BIG, dtype=np.int32)

    # fill sparse-ish candidate costs
    for i, key in enumerate(a):
        if not key:
            continue
        hits = process.extract(key, b, scorer=scorer, limit=min(top_k, m))
        for _, score, j in hits:
            if score >= min_score:
                cost[i, j] = 100 - int(score)

    # allow dummy-dummy = 0 so padding doesn't dominate
    if n < N and m < N:
        cost[n:, m:] = 0

    row_ind, col_ind = linear_sum_assignment(cost)

    ffq["matched_food_item"] = None
    ffq["match_score"] = 0
    ffq["match_warning"] = None
    for col in keep_cols:
        ffq[col] = None

    for r, c in zip(row_ind, col_ind):
        if r >= n:
            continue
        if c >= m:
            ffq.at[r, "match_warning"] = "unmatched"
            continue

        this_cost = cost[r, c]
        if this_cost >= BIG:
            ffq.at[r, "match_warning"] = "unmatched"
            continue

        score = 100 - int(this_cost)
        ffq.at[r, "matched_food_item"] = mapping.at[c, map_item_col]
        ffq.at[r, "match_score"] = score

        for col in keep_cols:
            ffq.at[r, col] = mapping.at[c, col] if col in mapping.columns else None

    ffq.drop(columns=["_key"], inplace=True)
    return ffq


# =========================
# USAGE
# =========================
# ffq_item is already a DataFrame in memory
# portion is a CSV on disk
portion = pd.read_csv(portion_path)  # <-- change path

# optional: ensure expected columns exist
# print(ffq_item.columns)
# print(portion.columns)

ffq_item_merged = one_to_one_fuzzy_merge(
    ffq_df=ffq_item,
    map_df=portion,
    ffq_item_col="item",
    map_item_col="food_item",
    keep_cols=["food_group", "portion"],
    min_score=85,   # stricter: 90-95
    top_k=30,       # larger = more accurate but slower
)

# Inspect unmatched / low score
unmatched = ffq_item_merged[ffq_item_merged["match_warning"].notna()]
print("Unmatched:", len(unmatched))
print(unmatched[["item", "match_warning", "match_score"]].head(30))


Unmatched: 109
                                  item match_warning  match_score
0                            ID Number     unmatched            0
1               Sugar in Teaspoons/day     unmatched            0
2                               filler     unmatched            0
3        Currently take multi-vitamin?     unmatched            0
4         How many multi-vit per week?     unmatched            0
5      Is this usually a gummy vitamin     unmatched            0
6       Labeled for Men, Women or Both     unmatched            0
7                       Centrum Silver     unmatched            0
8                              Centrum     unmatched            0
9                            One a day     unmatched            0
10       One a day Teens or equivalent     unmatched            0
11     Flintstones or Kids Multi equiv     unmatched            0
12                            Prenatal     unmatched            0
13                          Eye Health     unmatched         

In [20]:
ffq_item_merged[ffq_item_merged["food_group"].notna()]

,var,item,matched_food_item,match_score,match_warning,food_group,portion
30,calcium,Calcium,Orange juice (calcium or vitamin D fortified),100,None,Fruits,Small glass
59,c.flk.K,Corn Flakes,Corn,100,None,Vegetables,1 ear or 1/2 cup
62,natural.q,Granola,"Snack bars (Kind, Kashi, granola)",100,None,"Sweets, Baked Goods, Miscellaneous",1
68,skim.kids,Skim milk,Skim milk,100,None,Dairy Foods,8 oz glass
69,milk2,Milk - 1 or 2%,1 or 2% milk,100,None,Dairy Foods,8 oz glass
...,...,...,...,...,...,...,...
239,olive.oil,Olive oil added to food or bread,Olive oil added to food or bread,100,None,"Sweets, Baked Goods, Miscellaneous",1 Tbsp
240,mayo.d,Low fat or olive oil mayonnaise,Low-fat or olive oil mayonnaise,100,None,"Sweets, Baked Goods, Miscellaneous",1 Tbsp
241,mayo,Mayonnaise,Regular mayonnaise,100,None,"Sweets, Baked Goods, Miscellaneous",1 Tbsp
242,o.v,Salad dressing,Salad dressing,100,None,"Sweets, Baked Goods, Miscellaneous",1–2 Tbsp


In [21]:
# =========================================================
# 2) Helpers
# =========================================================
def _norm(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    return s

def _strip_parens(s: str) -> Tuple[str, List[str]]:
    mods = re.findall(r"\((.*?)\)", s)
    s2 = re.sub(r"\s*\(.*?\)\s*", " ", s)
    return _norm(s2), [_norm(m) for m in mods if _norm(m)]

def _split_as_context(s: str) -> Tuple[str, List[str]]:
    """
    Extract trailing "as a/an ..." context.
    e.g. "pork as a main dish" -> base="pork", ctx=["main dish"]
    """
    m = re.search(r"\bas\s+(an?\s+)?(.+)$", s, flags=re.I)
    if not m:
        return s, []
    base = _norm(s[:m.start()])
    ctx = _norm(m.group(2))
    return base, [ctx] if ctx else []

def _split_or_list(s: str) -> List[str]:
    s = _norm(s)
    s = re.sub(r"\band\/or\b", " or ", s, flags=re.I)
    parts = re.split(r"\s+\bor\b\s+", s, flags=re.I)
    return [_norm(p) for p in parts if _norm(p)]

def _split_slash(s: str) -> List[str]:
    if "/" not in s:
        return [s]
    parts = [p.strip() for p in s.split("/") if p.strip()]
    return parts if parts else [s]

def _split_sources(src_text: str) -> List[str]:
    """
    "chicken or turkey" / "beef, pork or lamb" / "chicken/turkey" -> tokens list
    Keep order as in text.
    """
    s = _norm(src_text).strip(" ,")
    if not s:
        return []
    # Turn commas into " " (we handle OR separately)
    s = re.sub(r"\s*,\s*", " ", s)
    out = []
    for chunk in _split_or_list(s):
        out.extend([x.strip() for x in chunk.split("/") if x.strip()])
    # clean punctuation
    out = [re.sub(r"[^\w\s-]", "", x).strip() for x in out]
    return [x for x in out if x]

def _singularize_simple(ent: str) -> str:
    # very light singularization for display: "dogs"->"dog", "hot dogs"->"hot dog"
    ent = ent.strip()
    if ent.endswith("s") and not ent.endswith("ss") and len(ent) > 3:
        return ent[:-1]
    return ent


# =========================================================
# 3) Food phrase support (minimal + var-derived)
# =========================================================
def build_food_phrases(df: pd.DataFrame) -> Set[str]:
    """
    Minimal seed + var-derived phrases so 'hot dog(s)' won't be split.
    This is NOT a big dictionary; it's a tiny guardrail.
    """
    phrases: Set[str] = {"hot dog", "hot dogs"}  # small seed for the known pain point

    for v in df["var"].astype(str).str.lower():
        v = v.strip()
        if not v or "." in v:
            continue
        toks = re.findall(r"[a-z]+", v)
        if len(toks) >= 2:
            p = " ".join(toks)
            phrases.add(p)
            phrases.add(p + "s")  # naive plural

    return phrases

def longest_phrase_match(text: str, phrases: Set[str]) -> str | None:
    t = text.lower()
    hits = [p for p in phrases if re.search(rf"\b{re.escape(p)}\b", t)]
    if not hits:
        return None
    return max(hits, key=len)


# =========================================================
# 4) Core parser: item -> list of records (expand OR/list)
# =========================================================
FORM_WORDS = {
    "pieces","piece","slices","slice","chunks","chunk","fillets","fillet",
    "sticks","stick","nuggets","nugget","patty","patties"
}

TYPE_MODIFIERS = {
    "regular",
    "lean",
    "extra lean",
    "low fat",
    "reduced fat",
    "full fat",
    "nonfat",
    "whole",
    "white",
    "dark",
    "light",
}

LEADING_MODIFIERS = {
    "canned","breaded","cooked","raw","fried","baked","grilled","smoked",
    "dark","white","whole","regular","lean","extra","reduced","full","low","nonfat",
    "fortified"
}

def _is_type_modifier_phrase(seg: str) -> bool:
    """
    seg 可能是 "regular" 或 "lean or extra lean"
    只要拆开后每个分量都属于 TYPE_MODIFIERS，就认为这是 type modifier segment
    """
    seg = _norm(seg).lower()
    if not seg:
        return False
    parts = []
    for p in _split_or_list(seg):
        parts.extend(_split_slash(p))
    parts = [_norm(x).lower() for x in parts if _norm(x)]
    return bool(parts) and all(x in TYPE_MODIFIERS for x in parts)


def _extract_type_mods_from_segment(seg: str) -> List[str]:
    """
    "lean or extra lean" -> ["lean", "extra lean"]
    """
    seg = _norm(seg).lower()
    out = []
    for p in _split_or_list(seg):
        out.extend([_norm(x).lower() for x in _split_slash(p) if _norm(x)])
    # 只保留真的 type mods
    return [x for x in out if x in TYPE_MODIFIERS]


def _extract_leading_modifiers(phrase: str) -> Tuple[str, List[str]]:
    """
    "Canned tuna" -> core="tuna", mods=["canned"]
    "Breaded fish cakes" -> core="fish cakes", mods=["breaded"]
    """
    toks = phrase.split()
    mods = []
    while toks and toks[0].lower() in LEADING_MODIFIERS:
        mods.append(toks.pop(0).lower())
    core = _norm(" ".join(toks)) if toks else phrase
    return core, mods

def clean_entity_not_start_with_digit(entity: str) -> str | None:
    """
    FFQ hard rule:
    - entity 不能以数字开头
    - 若以数字开头，尝试剥离前导数字
    - 若剥离后为空 / 仍非法，则返回 None（该 entity 丢弃）
    """
    if not entity:
        return None

    ent = entity.strip()

    # 如果不以数字开头，直接返回
    if not re.match(r"^\d", ent):
        return ent

    # 剥离前导数字（包括 1%, 2x, 100g 这种）
    ent2 = re.sub(r"^[\d\W_]+", "", ent).strip()

    # 剥离后为空 or 仍然不含字母 -> 判为非法
    if not ent2 or not re.search(r"[a-zA-Z]", ent2):
        return None

    return ent2

def parse_ffq_item(item: str, food_phrases: Set[str]) -> List[Dict[str, object]]:
    """
    Returns list of records:
      {"entity": str, "modifiers": List[str], "entity_modifiers": str}
    Expands:
      - comma-separated pure lists: "shrimp, crab, scallops, clams"
      - "or" lists: "sausage or bacon"
    Handles:
      - "X as a main dish" context -> modifier
      - parentheses -> modifiers
      - "chicken or turkey hot dogs" -> entity=hot dog, modifiers include chicken,turkey
    """
    item = _norm(item)
    if not item:
        return [{"entity": "", "modifiers": [], "entity_modifiers": ""}]

    # parentheses and "as a ..." context
    item_no_paren, paren_mods = _strip_parens(item)
    base, ctx_mods = _split_as_context(item_no_paren)

    # split by commas into segments
    segs = [_norm(x) for x in base.split(",") if _norm(x)]
    if not segs:
        segs = [base]

    first = segs[0]
    rest = segs[1:]

    records: List[Dict[str, object]] = []

    # ---- Case A: first segment contains a known food phrase (e.g., hot dog(s))
    phrase = longest_phrase_match(first, food_phrases)
    if phrase:
        ent = _singularize_simple(phrase.lower())

        # everything before the phrase is treated as source modifiers (chicken/turkey/beef etc.)
        pre = re.split(rf"\b{re.escape(phrase)}\b", first, flags=re.I)[0]
        src_mods = _split_sources(pre)

        mods = []
        mods.extend([m.lower() for m in src_mods if m])
        mods.extend([m.lower() for m in ctx_mods if m])
        mods.extend([m.lower() for m in paren_mods if m])

        combined = ent if not mods else ent + ", " + ", ".join(mods)
        records.append({"entity": ent, "modifiers": mods, "entity_modifiers": combined})

    else:
        # ---- Case B: treat first segment as entity phrase (with possible leading modifiers)
        core, lead_mods = _extract_leading_modifiers(first)

        # split "beef or lamb" inside core
        # also handle slash
        core_parts = []
        for p in _split_slash(core):
            core_parts.extend(_split_or_list(p))

        # if core itself looks like a pure list (single words), expand them
        entities = [c.lower() for c in core_parts if c] if core_parts else [core.lower()]

        # if the core is like "fish cakes" keep as-is; if it becomes "beef" etc still ok
        for ent in entities:
            ent = _norm(ent)
            mods = []
            mods.extend([m.lower() for m in lead_mods if m])
            mods.extend([m.lower() for m in ctx_mods if m])
            mods.extend([m.lower() for m in paren_mods if m])
            combined = ent if not mods else ent + ", " + ", ".join(mods)
            records.append({"entity": ent, "modifiers": mods, "entity_modifiers": combined})

    # ---- Now parse the remaining comma segments.
    # Heuristic:
    # - if segment is form word -> modifier to ALL existing records
    # - else segment is additional entity list -> add new records (no inheritance of mods by default)
        # ---- Now parse the remaining comma segments.
    # NEW rules:
    #   - FORM_WORDS -> modifiers applied to all existing records
    #   - TYPE_MODIFIERS segment (including "lean or extra lean") -> modifiers applied to all existing records
    #   - otherwise -> treat as additional entities (OR-expand)
    form_mods: List[str] = []
    type_mods: List[str] = []
    extra_entities: List[str] = []

    for seg in rest:
        seg_l = _norm(seg).lower().strip()

        # 1) form words like "pieces"
        if seg_l in FORM_WORDS:
            form_mods.append(seg_l)
            continue

        # 2) type modifier segment like "regular" / "lean or extra lean"
        if _is_type_modifier_phrase(seg_l):
            type_mods.extend(_extract_type_mods_from_segment(seg_l))
            continue

        # 3) otherwise: treat as extra entity list like "sausage or bacon"
        ents = []
        for part in _split_or_list(seg):
            ents.extend(_split_slash(part))
        ents = [e.lower().strip() for e in ents if e.strip()]
        extra_entities.extend(ents)

    # apply type + form modifiers to all existing records (IMPORTANT: affects entity_modifiers)
    if type_mods or form_mods:
        for r in records:
            r["modifiers"] = r["modifiers"] + type_mods + form_mods
            r["entity_modifiers"] = (
                r["entity"] if not r["modifiers"] else r["entity"] + ", " + ", ".join(r["modifiers"])
            )

    # add extra entity records (entity_modifiers must exist!)
    for ent in extra_entities:
        ent = _singularize_simple(_norm(ent))
        records.append({"entity": ent, "modifiers": [], "entity_modifiers": ent})


    # ---- Special: if the whole item is a pure comma list (shrimp, crab, scallops, clams)
    # The above already expands via extra_entities, but first seg would also create an entity record.
    # If it looks like ALL segments are short foods, we should treat first seg as entity only (no mods) and expand all.
    # ---- Special: pure food list (e.g., shrimp, crab, scallops, clams)
    if len(segs) >= 2:
        all_short = all(len(s.split()) <= 3 for s in segs)
        no_ctx = (not ctx_mods) and (not paren_mods)

    # NEW: if any segment is a known TYPE MODIFIER, this is NOT a pure list
        has_type_modifier = any(
            _norm(seg).lower() in TYPE_MODIFIERS for seg in segs[1:]
        )

        if all_short and no_ctx and not has_type_modifier:
            ents = []
            for seg in segs:
                ents.extend([e.lower().strip() for e in _split_or_list(seg) if e.strip()])
            records = [
                {
                    "entity": _singularize_simple(e),
                    "modifiers": [],
                    "entity_modifiers": _singularize_simple(e),
                }
                for e in ents
            ]
  

    return records


# =========================================================
# 5) Main: txt -> parse -> explode -> write rds
# =========================================================
def build_ffq_item_rds_from_df(
    ffq_item_df: pd.DataFrame,
    output_rds_path: str
) -> pd.DataFrame:
    """
    ffq_item_df: 已经在内存中的 DataFrame（必须包含 'var' 和 'item'）
    """
    df = ffq_item_df.copy()

    # 构建 food phrases（仍然基于 var）
    food_phrases = build_food_phrases(df)

    # parse item -> list of records
    parsed = df["item"].apply(lambda x: parse_ffq_item(x, food_phrases))

    # explode
    out = df.assign(_parsed=parsed).explode("_parsed", ignore_index=True)

    # unpack parsed fields
    out["entity"] = out["_parsed"].apply(lambda d: d["entity"])
    out["modifiers"] = out["_parsed"].apply(lambda d: d["modifiers"])
    out["entity_modifiers"] = out["_parsed"].apply(lambda d: d["entity_modifiers"])
    out = out.drop(columns=["_parsed"])

    # write RDS
    pyreadr.write_rds(output_rds_path, out)
    return out

assert {"var", "item"}.issubset(ffq_item_merged.columns)

if __name__ == "__main__":
    # ffq_item_merged 已经在内存中（例如前一步 fuzzy merge 的结果）
    # 确保它至少包含: var, item

    output_rds_path = output_rds_path  # 你的 rds 输出路径

    out_df = build_ffq_item_rds_from_df(
        ffq_item_df=ffq_item_merged,
        output_rds_path=output_rds_path
    )

    # sanity check
    check_vars = [
        "hotdog", "chix.dog", "sand.bf.ham",
        "pork", "beef", "tuna", "fr.fish.kids", "shrimp.ckd"
    ]

    print(
        out_df[out_df["var"].isin(check_vars)]
        [["var", "item", "entity_modifiers"]]
        .head(80)
    )



              var                                          item  \
216        hotdog                                 Beef Hot dogs   
217      chix.dog  Chicken or turkey hot dogs, sausage or bacon   
218      chix.dog  Chicken or turkey hot dogs, sausage or bacon   
219      chix.dog  Chicken or turkey hot dogs, sausage or bacon   
240   sand.bf.ham              Beef, pork or lamb as a sandwich   
241   sand.bf.ham              Beef, pork or lamb as a sandwich   
242   sand.bf.ham              Beef, pork or lamb as a sandwich   
243          pork                           Pork as a main dish   
244          beef                   Beef or lamb as a main dish   
245          beef                   Beef or lamb as a main dish   
246          tuna                                   Canned tuna   
247  fr.fish.kids                    Breaded fish cakes, pieces   
248  fr.fish.kids                    Breaded fish cakes, pieces   
249    shrimp.ckd                 Shrimp, crab, scallops, clam

In [22]:
out_df

,var,item,matched_food_item,match_score,match_warning,food_group,portion,entity,modifiers,entity_modifiers
0,id,ID Number,None,0,unmatched,None,None,id number,[],id number
1,sug,Sugar in Teaspoons/day,None,0,unmatched,None,None,sugar in teaspoons,[],sugar in teaspoons
2,sug,Sugar in Teaspoons/day,None,0,unmatched,None,None,day,[],day
3,blank2,filler,None,0,unmatched,None,None,filler,[],filler
4,multvit,Currently take multi-vitamin?,None,0,unmatched,None,None,currently take multi-vitamin?,[],currently take multi-vitamin?
...,...,...,...,...,...,...,...,...,...,...
381,as.ns,"Type of art sweetener, NutraSweet",None,0,unmatched,None,None,nutrasweet,[],nutrasweet
382,as.sl,"Type of art sweetener, Sweet N Low",None,0,unmatched,None,None,type of art sweetener,[],type of art sweetener
383,as.sl,"Type of art sweetener, Sweet N Low",None,0,unmatched,None,None,sweet n low,[],sweet n low
384,as.tr,"Type of art sweetener, Truvia",None,0,unmatched,None,None,type of art sweetener,[],type of art sweetener
